# Notebook 01 — Exploratory Data Analysis & Classical Test Theory

**Goal**: Perform EDA (demographics, missingness, endorsement distributions) and CTT analysis (item difficulty, item discrimination, reliability, floor/ceiling) for each sample separately, with statistical significance tests at every step.

- Sample 1 (Pretest): N = 1,035
- Sample 2 (Validation): N = 800
- IRT-relevant analyses use **real-author items only**; foils used for false-alarm scoring.

In [1]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import pingouin as pg

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

# Resolve project root
_MARKER = "data/processed/irt_pipeline/sample1_pretest.csv"
PROJECT_ROOT = os.path.abspath(os.getcwd())
while PROJECT_ROOT:
    if os.path.isfile(os.path.join(PROJECT_ROOT, _MARKER)):
        break
    _parent = os.path.dirname(PROJECT_ROOT)
    if _parent == PROJECT_ROOT:
        PROJECT_ROOT = os.path.abspath(os.getcwd())
        break
    PROJECT_ROOT = _parent

DATA_DIR = os.path.join(PROJECT_ROOT, "data/processed/irt_pipeline")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "data/processed/results/01_eda_ctt")
os.makedirs(RESULTS_DIR, exist_ok=True)

# Load data
s1 = pd.read_csv(os.path.join(DATA_DIR, "sample1_pretest.csv"))
s2 = pd.read_csv(os.path.join(DATA_DIR, "sample2_validation.csv"))
meta = pd.read_csv(os.path.join(DATA_DIR, "item_metadata.csv"))

DEMO_COLS = ['Submited', 'age', 'sex ', 'humanities or not', 'education and profession']
SOURCE_COL = 'source'
item_cols = [c for c in s1.columns if c not in DEMO_COLS + [SOURCE_COL]]
author_cols = meta.loc[meta['is_real_author'] == True, 'item_label'].tolist()
author_cols = [c for c in author_cols if c in item_cols]
foil_cols = meta.loc[meta['is_foil'] == True, 'item_label'].tolist()
foil_cols = [c for c in foil_cols if c in item_cols]

# Coerce items to numeric
for col in item_cols:
    s1[col] = pd.to_numeric(s1[col], errors='coerce')
    s2[col] = pd.to_numeric(s2[col], errors='coerce')

s1['age'] = pd.to_numeric(s1['age'], errors='coerce')
s2['age'] = pd.to_numeric(s2['age'], errors='coerce')

print(f"Sample 1: {len(s1)} participants, {len(author_cols)} authors, {len(foil_cols)} foils")
print(f"Sample 2: {len(s2)} participants, {len(author_cols)} authors, {len(foil_cols)} foils")

Sample 1: 1035 participants, 101 authors, 103 foils
Sample 2: 800 participants, 101 authors, 103 foils


---
## Part 1a: Exploratory Data Analysis
### 1.1 Demographics

In [2]:
def demographics_summary(df, name):
    print(f"\n{'='*60}")
    print(f"Demographics: {name} (N = {len(df)})")
    print(f"{'='*60}")
    
    age = df['age'].dropna()
    print(f"\nAge: M = {age.mean():.1f}, SD = {age.std():.1f}, "
          f"Median = {age.median():.0f}, Range = [{age.min():.0f}, {age.max():.0f}]")
    
    sex = df['sex '].value_counts()
    print(f"\nSex:")
    for k, v in sex.items():
        print(f"  {k}: {v} ({100*v/len(df):.1f}%)")
    
    hum = df['humanities or not'].value_counts()
    print(f"\nHumanities-connected:")
    for k, v in hum.items():
        print(f"  {k}: {v} ({100*v/len(df):.1f}%)")
    
    return {'age_mean': age.mean(), 'age_sd': age.std(), 'age_median': age.median(),
            'n': len(df), 'sex': sex.to_dict(), 'hum': hum.to_dict()}

d1 = demographics_summary(s1, 'Sample 1 (Pretest)')
d2 = demographics_summary(s2, 'Sample 2 (Validation)')


Demographics: Sample 1 (Pretest) (N = 1035)

Age: M = 27.5, SD = 9.7, Median = 26, Range = [1, 65]

Sex:
  F: 692 (66.9%)
  M: 343 (33.1%)

Humanities-connected:
  -: 667 (64.4%)
  +: 231 (22.3%)
  _: 3 (0.3%)

Demographics: Sample 2 (Validation) (N = 800)

Age: M = 28.0, SD = 10.0, Median = 27, Range = [1, 65]

Sex:
  F: 572 (71.5%)
  M: 228 (28.5%)

Humanities-connected:
  -: 800 (100.0%)


### 1.2 Between-sample demographic comparison (significance tests)

In [3]:
print("="*60)
print("Between-sample comparisons")
print("="*60)

# Age: Mann-Whitney U (non-parametric, robust to non-normal age distributions)
age1 = s1['age'].dropna()
age2 = s2['age'].dropna()
u_stat, u_p = stats.mannwhitneyu(age1, age2, alternative='two-sided')
# Effect size: rank-biserial r = 1 - 2U/(n1*n2)
r_rb = 1 - 2*u_stat / (len(age1)*len(age2))
print(f"\nAge (Mann-Whitney U):")
print(f"  U = {u_stat:.0f}, p = {u_p:.4f}, rank-biserial r = {r_rb:.3f}")
print(f"  S1 mean={age1.mean():.1f}, S2 mean={age2.mean():.1f}")

# Also Welch's t-test for comparison
t_stat, t_p = stats.ttest_ind(age1, age2, equal_var=False)
cohens_d = (age1.mean() - age2.mean()) / np.sqrt((age1.var() + age2.var()) / 2)
print(f"  Welch's t = {t_stat:.3f}, p = {t_p:.4f}, Cohen's d = {cohens_d:.3f}")

# Sex: Chi-square
sex_ct = pd.DataFrame({
    'S1': s1['sex '].value_counts(),
    'S2': s2['sex '].value_counts()
}).fillna(0)
chi2, chi_p, dof, expected = stats.chi2_contingency(sex_ct.values)
cramers_v = np.sqrt(chi2 / (sex_ct.values.sum() * (min(sex_ct.shape) - 1)))
print(f"\nSex (Chi-square):")
print(f"  chi2({dof}) = {chi2:.3f}, p = {chi_p:.4f}, Cramer's V = {cramers_v:.3f}")

# Humanities: Chi-square
hum_ct = pd.DataFrame({
    'S1': s1['humanities or not'].value_counts(),
    'S2': s2['humanities or not'].value_counts()
}).fillna(0)
chi2h, chi_ph, dofh, _ = stats.chi2_contingency(hum_ct.values)
cramers_vh = np.sqrt(chi2h / (hum_ct.values.sum() * (min(hum_ct.shape) - 1)))
print(f"\nHumanities-connected (Chi-square):")
print(f"  chi2({dofh}) = {chi2h:.3f}, p = {chi_ph:.4f}, Cramer's V = {cramers_vh:.3f}")

Between-sample comparisons

Age (Mann-Whitney U):
  U = 397976, p = 0.1543, rank-biserial r = 0.039
  S1 mean=27.5, S2 mean=28.0
  Welch's t = -1.131, p = 0.2581, Cohen's d = -0.053

Sex (Chi-square):
  chi2(1) = 4.319, p = 0.0377, Cramer's V = 0.049

Humanities-connected (Chi-square):
  chi2(2) = 240.910, p = 0.0000, Cramer's V = 0.376


### 1.3 Missingness analysis

In [4]:
def missingness_report(df, cols, name):
    miss = df[cols].isna().mean() * 100
    total_miss = df[cols].isna().sum().sum()
    total_cells = df[cols].size
    
    print(f"\n{name}:")
    print(f"  Overall: {total_miss:,} / {total_cells:,} ({100*total_miss/total_cells:.2f}%)")
    print(f"  Items with >5% missing: {(miss > 5).sum()}")
    print(f"  Items with >10% missing: {(miss > 10).sum()}")
    
    top_miss = miss.nlargest(5)
    if top_miss.max() > 0:
        print(f"  Top-5 missing:")
        for item, pct in top_miss.items():
            print(f"    {item}: {pct:.1f}%")
    return miss

print("="*60)
print("Missingness Analysis (Author Items)")
print("="*60)
miss1_auth = missingness_report(s1, author_cols, 'Sample 1 (Pretest) — Authors')
miss2_auth = missingness_report(s2, author_cols, 'Sample 2 (Validation) — Authors')

print("\n" + "="*60)
print("Missingness Analysis (Foil Items)")
print("="*60)
miss1_foil = missingness_report(s1, foil_cols, 'Sample 1 (Pretest) — Foils')
miss2_foil = missingness_report(s2, foil_cols, 'Sample 2 (Validation) — Foils')

Missingness Analysis (Author Items)

Sample 1 (Pretest) — Authors:
  Overall: 586 / 104,535 (0.56%)
  Items with >5% missing: 1
  Items with >10% missing: 1
  Top-5 missing:
    Yuri Tsypkin: 56.5%
    Ian Fleming: 0.1%
    Khaled Hosseini: 0.0%
    Donna Tartt: 0.0%
    Archibald Cronin: 0.0%



Sample 2 (Validation) — Authors:
  Overall: 2 / 80,800 (0.00%)
  Items with >5% missing: 0
  Items with >10% missing: 0
  Top-5 missing:
    Ian Fleming: 0.1%
    Lawrense Stern: 0.1%
    Khaled Hosseini: 0.0%
    Donna Tartt: 0.0%
    Archibald Cronin: 0.0%

Missingness Analysis (Foil Items)



Sample 1 (Pretest) — Foils:
  Overall: 0 / 106,605 (0.00%)
  Items with >5% missing: 0
  Items with >10% missing: 0

Sample 2 (Validation) — Foils:
  Overall: 582 / 82,400 (0.71%)
  Items with >5% missing: 1
  Items with >10% missing: 1
  Top-5 missing:
    Andrea Segre fill 93: 72.0%
    Gerrit HoogenbuM fill1: 0.2%
    Vasily Uzun fill 98: 0.2%
    Yakushkina Gilyan fill 83: 0.1%
    Natalya Shagaida fill 94: 0.1%


### 1.4 Endorsement rate distributions (authors only)

In [5]:
endorse1 = s1[author_cols].mean()
endorse2 = s2[author_cols].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(endorse1, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_title(f'Sample 1 (N={len(s1)}): Author Endorsement Rates')
axes[0].set_xlabel('Endorsement Rate')
axes[0].set_ylabel('Number of Items')
axes[0].axvline(endorse1.mean(), color='red', linestyle='--', label=f'Mean={endorse1.mean():.3f}')
axes[0].legend()

axes[1].hist(endorse2, bins=30, edgecolor='black', alpha=0.7, color='darkorange')
axes[1].set_title(f'Sample 2 (N={len(s2)}): Author Endorsement Rates')
axes[1].set_xlabel('Endorsement Rate')
axes[1].set_ylabel('Number of Items')
axes[1].axvline(endorse2.mean(), color='red', linestyle='--', label=f'Mean={endorse2.mean():.3f}')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'endorsement_distributions.png'), dpi=150)
plt.show()

# Paired comparison of endorsement rates across samples
t_endorse, p_endorse = stats.ttest_rel(endorse1, endorse2)
print(f"\nPaired t-test on item endorsement rates (S1 vs S2):")
print(f"  t({len(endorse1)-1}) = {t_endorse:.3f}, p = {p_endorse:.4f}")
print(f"  S1 mean = {endorse1.mean():.3f}, S2 mean = {endorse2.mean():.3f}")

# Correlation of endorsement rates across samples
r_endorse, p_r = stats.pearsonr(endorse1, endorse2)
print(f"  Pearson r = {r_endorse:.4f}, p = {p_r:.2e}")


Paired t-test on item endorsement rates (S1 vs S2):
  t(100) = 3.602, p = 0.0005
  S1 mean = 0.513, S2 mean = 0.416
  Pearson r = 0.5504, p = 2.48e-09


/tmp/ipykernel_38793/2717562665.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 1.5 False alarm rate distributions (foils)

In [6]:
fa1 = s1[foil_cols].mean()
fa2 = s2[foil_cols].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(fa1, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_title(f'Sample 1: Foil False Alarm Rates')
axes[0].set_xlabel('False Alarm Rate')
axes[0].set_ylabel('Number of Items')

axes[1].hist(fa2, bins=30, edgecolor='black', alpha=0.7, color='darkorange')
axes[1].set_title(f'Sample 2: Foil False Alarm Rates')
axes[1].set_xlabel('False Alarm Rate')
axes[1].set_ylabel('Number of Items')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'false_alarm_distributions.png'), dpi=150)
plt.show()

print(f"Sample 1 FA: M = {fa1.mean():.3f}, SD = {fa1.std():.3f}, Max = {fa1.max():.3f}")
print(f"Sample 2 FA: M = {fa2.mean():.3f}, SD = {fa2.std():.3f}, Max = {fa2.max():.3f}")

Sample 1 FA: M = 0.044, SD = 0.040, Max = 0.219
Sample 2 FA: M = 0.136, SD = 0.217, Max = 0.889


/tmp/ipykernel_38793/42985998.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Part 1b: Classical Test Theory
### 2.1 Item difficulty (endorsement rates, authors only)

In [7]:
def ctt_item_analysis(df, author_cols, foil_cols, name):
    """Compute full CTT item analysis for one sample."""
    results = {}
    
    # Item difficulty = endorsement rate p
    p_vals = df[author_cols].mean()
    results['difficulty'] = p_vals
    
    # Total score (sum of author items)
    total = df[author_cols].sum(axis=1)
    
    # Item discrimination: corrected point-biserial r (item-rest correlation)
    rpb = {}
    rpb_p = {}
    for col in author_cols:
        item = df[col].dropna()
        rest = df[author_cols].drop(columns=[col]).sum(axis=1).loc[item.index]
        if item.std() == 0 or rest.std() == 0:
            rpb[col] = 0.0
            rpb_p[col] = 1.0
            continue
        r, p = stats.pointbiserialr(item, rest)
        rpb[col] = r
        rpb_p[col] = p
    
    results['rpb'] = pd.Series(rpb)
    results['rpb_pval'] = pd.Series(rpb_p)
    
    # Floor/ceiling items
    floor_items = p_vals[p_vals < 0.05].index.tolist()
    ceiling_items = p_vals[p_vals > 0.95].index.tolist()
    results['floor'] = floor_items
    results['ceiling'] = ceiling_items
    
    # Classical ART scoring
    hits = df[author_cols].sum(axis=1)
    false_alarms = df[foil_cols].sum(axis=1)
    standard_score = hits - false_alarms
    penalty2_score = hits - 2 * false_alarms
    
    results['scoring'] = pd.DataFrame({
        'hits': hits,
        'false_alarms': false_alarms,
        'standard_score': standard_score,
        'penalty2_score': penalty2_score
    })
    
    # Print
    print(f"\n{'='*60}")
    print(f"CTT Item Analysis: {name}")
    print(f"{'='*60}")
    
    print(f"\nItem Difficulty (endorsement rate):")
    print(f"  Mean p = {p_vals.mean():.3f}, SD = {p_vals.std():.3f}")
    print(f"  Range: [{p_vals.min():.3f}, {p_vals.max():.3f}]")
    
    print(f"\nItem Discrimination (corrected point-biserial r):")
    print(f"  Mean r_pb = {results['rpb'].mean():.3f}, SD = {results['rpb'].std():.3f}")
    print(f"  Range: [{results['rpb'].min():.3f}, {results['rpb'].max():.3f}]")
    n_poor = (results['rpb'] < 0.20).sum()
    n_sig = (results['rpb_pval'] < 0.05).sum()
    print(f"  Items with r_pb < 0.20 (poor discrimination): {n_poor}")
    print(f"  Items with significant r_pb (p < .05): {n_sig} / {len(author_cols)}")
    
    print(f"\nFloor items (p < 0.05): {len(floor_items)}")
    for it in floor_items:
        print(f"    {it}: p = {p_vals[it]:.3f}")
    
    print(f"\nCeiling items (p > 0.95): {len(ceiling_items)}")
    for it in ceiling_items:
        print(f"    {it}: p = {p_vals[it]:.3f}")
    
    scoring = results['scoring']
    print(f"\nClassical ART Scores:")
    print(f"  Hits:       M = {scoring['hits'].mean():.1f}, SD = {scoring['hits'].std():.1f}")
    print(f"  False alarms: M = {scoring['false_alarms'].mean():.1f}, SD = {scoring['false_alarms'].std():.1f}")
    print(f"  Standard (H-FA): M = {scoring['standard_score'].mean():.1f}, SD = {scoring['standard_score'].std():.1f}")
    print(f"  Penalty-2 (H-2FA): M = {scoring['penalty2_score'].mean():.1f}, SD = {scoring['penalty2_score'].std():.1f}")
    
    # Hits-FA correlation
    r_hfa, p_hfa = stats.pearsonr(scoring['hits'], scoring['false_alarms'])
    print(f"  Hits-FA correlation: r = {r_hfa:.3f}, p = {p_hfa:.4f}")
    
    return results

ctt1 = ctt_item_analysis(s1, author_cols, foil_cols, 'Sample 1 (Pretest)')
ctt2 = ctt_item_analysis(s2, author_cols, foil_cols, 'Sample 2 (Validation)')


CTT Item Analysis: Sample 1 (Pretest)

Item Difficulty (endorsement rate):
  Mean p = 0.513, SD = 0.258
  Range: [0.043, 0.981]

Item Discrimination (corrected point-biserial r):
  Mean r_pb = 0.459, SD = 0.148
  Range: [0.109, 0.695]
  Items with r_pb < 0.20 (poor discrimination): 7
  Items with significant r_pb (p < .05): 101 / 101

Floor items (p < 0.05): 1
    Yustein Gordier: p = 0.043

Ceiling items (p > 0.95): 7
    Charles Dickens: p = 0.954
    Jack London: p = 0.977
    Arthur Conan Doyle: p = 0.972
    Alexandre Dumas: p = 0.962
    Agatha Christie: p = 0.971
    Ray Bradbury: p = 0.957
    Jules Verne: p = 0.981

Classical ART Scores:
  Hits:       M = 51.8, SD = 21.0
  False alarms: M = 4.5, SD = 7.3
  Standard (H-FA): M = 47.3, SD = 21.2
  Penalty-2 (H-2FA): M = 42.8, SD = 23.8
  Hits-FA correlation: r = 0.147, p = 0.0000



CTT Item Analysis: Sample 2 (Validation)

Item Difficulty (endorsement rate):
  Mean p = 0.416, SD = 0.307
  Range: [0.009, 0.984]

Item Discrimination (corrected point-biserial r):
  Mean r_pb = 0.379, SD = 0.205
  Range: [0.041, 0.691]
  Items with r_pb < 0.20 (poor discrimination): 25
  Items with significant r_pb (p < .05): 91 / 101

Floor items (p < 0.05): 17
    Dmutry Gluhovsky: p = 0.020
    Zakhar Prilepin: p = 0.019
    Alan Milne: p = 0.029
    Ethel Lilian Voynich: p = 0.019
    Harper Lee: p = 0.035
    Dan Bruwn: p = 0.037
    Neil Gaiman: p = 0.026
    Ayn Rand: p = 0.015
    Daniel Keyes: p = 0.040
    Jules Verne: p = 0.029
    Lawrense Stern: p = 0.020
    Dina Rubina: p = 0.035
    Bernard Shaw: p = 0.011
    Dmitry Bykov: p = 0.009
    Mikhail Elizarov: p = 0.028
    Andrey Belyanin: p = 0.011
    Yuri Tsypkin: p = 0.009

Ceiling items (p > 0.95): 7
    Jack London: p = 0.970
    Arthur Conan Doyle: p = 0.970
    John R.R. Tolkien: p = 0.958
    Alexandre Dumas: p 

### 2.2 Internal consistency: Cronbach's alpha

In [8]:
def cronbach_alpha_bootstrap(df, cols, n_boot=1000, ci=0.95, seed=42):
    """Compute Cronbach's alpha with bootstrap CI."""
    data = df[cols].dropna()
    k = len(cols)
    
    def alpha_from_data(d):
        item_vars = d.var(axis=0, ddof=1)
        total_var = d.sum(axis=1).var(ddof=1)
        if total_var == 0:
            return np.nan
        return (k / (k - 1)) * (1 - item_vars.sum() / total_var)
    
    alpha_obs = alpha_from_data(data)
    
    rng = np.random.RandomState(seed)
    boot_alphas = []
    n = len(data)
    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        boot_data = data.iloc[idx]
        boot_alphas.append(alpha_from_data(boot_data))
    
    boot_alphas = np.array(boot_alphas)
    lo = np.percentile(boot_alphas, 100 * (1 - ci) / 2)
    hi = np.percentile(boot_alphas, 100 * (1 + ci) / 2)
    
    return alpha_obs, lo, hi, boot_alphas

print("="*60)
print("Internal Consistency: Cronbach's Alpha")
print("="*60)

a1, lo1, hi1, boot1 = cronbach_alpha_bootstrap(s1, author_cols)
a2, lo2, hi2, boot2 = cronbach_alpha_bootstrap(s2, author_cols)

print(f"\nSample 1: alpha = {a1:.4f}, 95% CI = [{lo1:.4f}, {hi1:.4f}]")
print(f"Sample 2: alpha = {a2:.4f}, 95% CI = [{lo2:.4f}, {hi2:.4f}]")

# Test if alpha > 0.70
prop_above_70_s1 = np.mean(boot1 > 0.70)
prop_above_70_s2 = np.mean(boot2 > 0.70)
print(f"\nP(alpha > 0.70) — Sample 1: {prop_above_70_s1:.4f}")
print(f"P(alpha > 0.70) — Sample 2: {prop_above_70_s2:.4f}")

# Feldt test for comparing two independent alphas
n1, n2, k = len(s1), len(s2), len(author_cols)
F_feldt = (1 - a2) / (1 - a1)
df1_feldt = n1 - 1
df2_feldt = n2 - 1
p_feldt = 2 * min(stats.f.cdf(F_feldt, df1_feldt, df2_feldt),
                  1 - stats.f.cdf(F_feldt, df1_feldt, df2_feldt))
print(f"\nFeldt test (alpha_S1 vs alpha_S2):")
print(f"  F = {F_feldt:.4f}, p = {p_feldt:.4f}")

Internal Consistency: Cronbach's Alpha



Sample 1: alpha = 0.9675, 95% CI = [0.9637, 0.9706]
Sample 2: alpha = 0.9572, 95% CI = [0.9539, 0.9605]

P(alpha > 0.70) — Sample 1: 1.0000
P(alpha > 0.70) — Sample 2: 1.0000

Feldt test (alpha_S1 vs alpha_S2):
  F = 1.3174, p = 0.0000


### 2.3 Split-half reliability

In [9]:
def split_half_reliability(df, cols, n_splits=100, seed=42):
    """Compute split-half reliability with Spearman-Brown correction."""
    data = df[cols].dropna()
    rng = np.random.RandomState(seed)
    r_halves = []
    
    for _ in range(n_splits):
        idx = rng.permutation(len(cols))
        half1_cols = [cols[i] for i in idx[:len(cols)//2]]
        half2_cols = [cols[i] for i in idx[len(cols)//2:]]
        
        s_h1 = data[half1_cols].sum(axis=1)
        s_h2 = data[half2_cols].sum(axis=1)
        
        r, _ = stats.pearsonr(s_h1, s_h2)
        r_halves.append(r)
    
    r_half = np.mean(r_halves)
    sb = 2 * r_half / (1 + r_half)  # Spearman-Brown
    
    return r_half, sb, np.std(r_halves)

print("="*60)
print("Split-Half Reliability (Spearman-Brown corrected)")
print("="*60)

rh1, sb1, sd1 = split_half_reliability(s1, author_cols)
rh2, sb2, sd2 = split_half_reliability(s2, author_cols)

print(f"\nSample 1: r_half = {rh1:.4f} (SD = {sd1:.4f}), Spearman-Brown = {sb1:.4f}")
print(f"Sample 2: r_half = {rh2:.4f} (SD = {sd2:.4f}), Spearman-Brown = {sb2:.4f}")

Split-Half Reliability (Spearman-Brown corrected)



Sample 1: r_half = 0.9425 (SD = 0.0065), Spearman-Brown = 0.9704
Sample 2: r_half = 0.9263 (SD = 0.0098), Spearman-Brown = 0.9617


### 2.4 Ferguson's delta

In [10]:
def fergusons_delta(total_scores, k):
    """Compute Ferguson's delta (distribution discrimination index)."""
    n = len(total_scores)
    freq = total_scores.value_counts().sort_index()
    sum_fi_sq = (freq ** 2).sum()
    delta = (n**2 - sum_fi_sq) / (n**2 - n**2 / (k + 1))
    return delta

total1 = s1[author_cols].sum(axis=1)
total2 = s2[author_cols].sum(axis=1)

fd1 = fergusons_delta(total1, len(author_cols))
fd2 = fergusons_delta(total2, len(author_cols))

print("="*60)
print("Ferguson's Delta (distribution discrimination)")
print("="*60)
print(f"  Sample 1: delta = {fd1:.4f}")
print(f"  Sample 2: delta = {fd2:.4f}")
print(f"  (Values > 0.90 indicate good score discrimination)")

Ferguson's Delta (distribution discrimination)
  Sample 1: delta = 0.9960
  Sample 2: delta = 0.9923
  (Values > 0.90 indicate good score discrimination)


### 2.5 Item-level CTT summary table

In [11]:
def build_ctt_table(ctt_result, sample_name):
    tbl = pd.DataFrame({
        'item': ctt_result['difficulty'].index,
        'difficulty_p': ctt_result['difficulty'].values,
        'discrimination_rpb': ctt_result['rpb'].values,
        'rpb_pvalue': ctt_result['rpb_pval'].values,
        'rpb_significant': ctt_result['rpb_pval'].values < 0.05
    })
    tbl['floor'] = tbl['difficulty_p'] < 0.05
    tbl['ceiling'] = tbl['difficulty_p'] > 0.95
    tbl = tbl.sort_values('discrimination_rpb', ascending=False)
    return tbl

tbl1 = build_ctt_table(ctt1, 'Sample 1')
tbl2 = build_ctt_table(ctt2, 'Sample 2')

# Save CTT tables
tbl1.to_csv(os.path.join(RESULTS_DIR, 'ctt_items_sample1.csv'), index=False)
tbl2.to_csv(os.path.join(RESULTS_DIR, 'ctt_items_sample2.csv'), index=False)

print(f"Top-10 discriminating items (Sample 1):")
print(tbl1[['item', 'difficulty_p', 'discrimination_rpb', 'rpb_pvalue']].head(10).to_string(index=False))
print(f"\nTop-10 discriminating items (Sample 2):")
print(tbl2[['item', 'difficulty_p', 'discrimination_rpb', 'rpb_pvalue']].head(10).to_string(index=False))

Top-10 discriminating items (Sample 1):
                  item  difficulty_p  discrimination_rpb    rpb_pvalue
           John Fowles      0.497585            0.694669 5.744005e-150
     William Thackeray      0.508213            0.687190 1.585517e-145
      Somerset Maugham      0.654106            0.685360 1.845717e-144
         Milorad Pavic      0.334300            0.675182 1.135369e-138
           Dina Rubina      0.540097            0.670414 4.871631e-136
          Yury  Olesha      0.568116            0.657425 4.123104e-129
        Victor Erofeev      0.444444            0.640708 1.095733e-120
           Isaac Babel      0.579710            0.639409 4.706148e-120
        Leonid Andreev      0.434783            0.638892 8.388088e-120
Gabriel Garsia Marquez      0.760386            0.627880 1.435155e-114

Top-10 discriminating items (Sample 2):
                  item  difficulty_p  discrimination_rpb    rpb_pvalue
           John Fowles       0.52500            0.690812 1.658667e-

### 2.6 Discrimination scatter plot (Sample 1 vs Sample 2)

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Difficulty scatter
axes[0].scatter(ctt1['difficulty'], ctt2['difficulty'], alpha=0.5, s=20)
axes[0].plot([0, 1], [0, 1], 'r--', alpha=0.5)
axes[0].set_xlabel('Sample 1 — p (difficulty)')
axes[0].set_ylabel('Sample 2 — p (difficulty)')
axes[0].set_title('Item Difficulty: S1 vs S2')
r_d, p_d = stats.pearsonr(ctt1['difficulty'], ctt2['difficulty'])
axes[0].annotate(f'r = {r_d:.3f}, p = {p_d:.2e}', xy=(0.05, 0.92),
                 xycoords='axes fraction', fontsize=10)

# Discrimination scatter
axes[1].scatter(ctt1['rpb'], ctt2['rpb'], alpha=0.5, s=20)
lim = [min(ctt1['rpb'].min(), ctt2['rpb'].min()) - 0.05,
       max(ctt1['rpb'].max(), ctt2['rpb'].max()) + 0.05]
axes[1].plot(lim, lim, 'r--', alpha=0.5)
axes[1].set_xlabel('Sample 1 — r_pb (discrimination)')
axes[1].set_ylabel('Sample 2 — r_pb (discrimination)')
axes[1].set_title('Item Discrimination: S1 vs S2')
r_disc, p_disc = stats.pearsonr(ctt1['rpb'], ctt2['rpb'])
axes[1].annotate(f'r = {r_disc:.3f}, p = {p_disc:.2e}', xy=(0.05, 0.92),
                 xycoords='axes fraction', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'ctt_parameter_comparison.png'), dpi=150)
plt.show()

print(f"Difficulty correlation: r = {r_d:.4f}, p = {p_d:.2e}")
print(f"Discrimination correlation: r = {r_disc:.4f}, p = {p_disc:.2e}")

Difficulty correlation: r = 0.5504, p = 2.48e-09
Discrimination correlation: r = 0.3872, p = 6.33e-05


/tmp/ipykernel_38793/3212491781.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 2.7 Score distribution plots

In [13]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, (sample, ctt, name, color) in enumerate([
    (s1, ctt1, 'Sample 1', 'steelblue'),
    (s2, ctt2, 'Sample 2', 'darkorange')
]):
    sc = ctt['scoring']
    axes[i, 0].hist(sc['hits'], bins=40, edgecolor='black', alpha=0.7, color=color)
    axes[i, 0].set_title(f'{name}: Hits Distribution')
    axes[i, 0].set_xlabel('Hits (Author Endorsements)')
    
    axes[i, 1].hist(sc['standard_score'], bins=40, edgecolor='black', alpha=0.7, color=color)
    axes[i, 1].set_title(f'{name}: Standard Score (Hits - FA)')
    axes[i, 1].set_xlabel('Standard ART Score')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'score_distributions.png'), dpi=150)
plt.show()

/tmp/ipykernel_38793/3569428578.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 2.8 Summary

In [14]:
summary = pd.DataFrame({
    'Metric': [
        'N', 'Author Items', 'Foil Items',
        'Mean difficulty p', 'Mean discrimination r_pb',
        'Items r_pb < 0.20', 'Floor items (p < .05)', 'Ceiling items (p > .95)',
        'Cronbach alpha', 'Alpha 95% CI',
        'Split-half (Spearman-Brown)',
        'Ferguson delta',
        'Mean Hits', 'Mean FA', 'Mean Standard Score'
    ],
    'Sample 1': [
        len(s1), len(author_cols), len(foil_cols),
        f"{ctt1['difficulty'].mean():.3f}", f"{ctt1['rpb'].mean():.3f}",
        f"{(ctt1['rpb'] < 0.20).sum()}", f"{len(ctt1['floor'])}", f"{len(ctt1['ceiling'])}",
        f"{a1:.4f}", f"[{lo1:.4f}, {hi1:.4f}]",
        f"{sb1:.4f}",
        f"{fd1:.4f}",
        f"{ctt1['scoring']['hits'].mean():.1f}",
        f"{ctt1['scoring']['false_alarms'].mean():.1f}",
        f"{ctt1['scoring']['standard_score'].mean():.1f}"
    ],
    'Sample 2': [
        len(s2), len(author_cols), len(foil_cols),
        f"{ctt2['difficulty'].mean():.3f}", f"{ctt2['rpb'].mean():.3f}",
        f"{(ctt2['rpb'] < 0.20).sum()}", f"{len(ctt2['floor'])}", f"{len(ctt2['ceiling'])}",
        f"{a2:.4f}", f"[{lo2:.4f}, {hi2:.4f}]",
        f"{sb2:.4f}",
        f"{fd2:.4f}",
        f"{ctt2['scoring']['hits'].mean():.1f}",
        f"{ctt2['scoring']['false_alarms'].mean():.1f}",
        f"{ctt2['scoring']['standard_score'].mean():.1f}"
    ]
})

print("\n" + "="*60)
print("CTT SUMMARY TABLE")
print("="*60)
print(summary.to_string(index=False))

summary.to_csv(os.path.join(RESULTS_DIR, 'ctt_summary.csv'), index=False)
print(f"\nSaved to: {RESULTS_DIR}")


CTT SUMMARY TABLE
                     Metric         Sample 1         Sample 2
                          N             1035              800
               Author Items              101              101
                 Foil Items              103              103
          Mean difficulty p            0.513            0.416
   Mean discrimination r_pb            0.459            0.379
          Items r_pb < 0.20                7               25
      Floor items (p < .05)                1               17
    Ceiling items (p > .95)                7                7
             Cronbach alpha           0.9675           0.9572
               Alpha 95% CI [0.9637, 0.9706] [0.9539, 0.9605]
Split-half (Spearman-Brown)           0.9704           0.9617
             Ferguson delta           0.9960           0.9923
                  Mean Hits             51.8             42.0
                    Mean FA              4.5             14.0
        Mean Standard Score             47.3       